# while_loops
`while` loops are for repetition driven by state, not by a fixed number of items. They are useful in polling, retries, streaming reads, and state machines, but they are also where control-flow bugs can become infinite loops, runaway retries, or CPU waste.

## 1. Problem First
Why does this matter in real systems?
- Retry logic often loops until success or a limit is hit.
- Polling loops wait for external state changes.
- A bad exit condition can burn CPU or spam external systems.

In [1]:
attempt = 0
while attempt < 3:
    print(f"attempt={attempt}")
    attempt += 1

attempt=0
attempt=1
attempt=2


## 2. Minimal Concept
Core syntax:
- `while condition:` repeats as long as the condition stays truthy.
- You must change state so the condition can eventually become false.
- `while` is best when the stop condition is state-based, not collection-based.

## 3. Mental Model
How Python actually behaves
A `while` loop checks the condition before every iteration. If the condition is true, the body runs. If the condition becomes false, the loop exits. The loop is entirely controlled by evolving state, so correctness depends on whether that state changes the way you think it does.

In [2]:
remaining = 3
while remaining:
    print(remaining)
    remaining -= 1
print("done")

3
2
1
done


## 4. Internal Mechanics
A `while` loop compiles into repeated condition checks plus jumps back to the loop start. Python does not protect you from bad exit conditions; it will keep going until the condition becomes false or execution is interrupted.

In [3]:
import dis

def countdown(n):
    while n > 0:
        n -= 1
    return n

dis.dis(countdown)
print(countdown(3))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (n)
              4 LOAD_CONST               1 (0)
              6 COMPARE_OP              68 (>)
             10 POP_JUMP_IF_FALSE       11 (to 34)

  5     >>   12 LOAD_FAST                0 (n)
             14 LOAD_CONST               2 (1)
             16 BINARY_OP               23 (-=)
             20 STORE_FAST               0 (n)

  4          22 LOAD_FAST                0 (n)
             24 LOAD_CONST               1 (0)
             26 COMPARE_OP              68 (>)
             30 POP_JUMP_IF_FALSE        1 (to 34)
             32 JUMP_BACKWARD           11 (to 12)

  6     >>   34 LOAD_FAST                0 (n)
             36 RETURN_VALUE
0


## 5. Edge Cases
Where it breaks:
- Forgetting to update state creates infinite loops.
- Updating the wrong variable creates fake progress.
- `while True` without a reliable break condition becomes dangerous fast.
- External state may never change, so polling loops need limits or timeouts.

In [4]:
counter = 3
steps = 0
while counter > 0 and steps < 5:
    print(counter)
    steps += 1
print("counter never changed")

3
3
3
3
3
counter never changed


## 6. Performance Thinking
- A `while` loop is O(k) in the number of iterations it actually performs.
- Polling without sleep or backoff can waste CPU.
- Retry loops should cap attempts and often apply backoff.
- The expensive part is usually the work inside the loop, not the loop syntax itself.

## 7. Real Use Case
A client retries a request until it succeeds or the retry budget is exhausted.

In [5]:
success = False
attempt = 0
max_attempts = 3
while not success and attempt < max_attempts:
    attempt += 1
    print(f"trying attempt {attempt}")
    if attempt == 3:
        success = True
print(success)

trying attempt 1
trying attempt 2
trying attempt 3
True


## 8. Anti-Pattern
What beginners do wrong:
- Use `while` where a `for` loop over known items would be simpler.
- Write retry loops with no upper bound.
- Hide loop progress in too many mutable variables.

In [6]:
index = 0
items = ["a", "b", "c"]
while index < len(items):
    print(items[index])
    index += 1

for item in items:
    print(item)

a
b
c
a
b
c


## 9. Interview Signals
Questions FAANG asks:
- When should you use `while` instead of `for`?
- What makes a retry loop safe?
- How do infinite loops happen in Python?
- What safeguards belong in polling logic?

## 10. Exercise (Non-trivial)
Design a polling loop for a background job. It should stop when the job completes, when a timeout is reached, or when the service starts returning fatal errors. Make the state transitions explicit and keep the loop debuggable under incident pressure.

In [7]:
def wait_for_job(max_attempts):
    attempt = 0
    status = "pending"
    while status == "pending":
        attempt += 1
        if attempt >= max_attempts:
            return "timeout"
    return status

# Task:
# 1. Add a success path and a fatal error path.
# 2. Prevent runaway looping.
# 3. Explain which state variables control exit behavior.